In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

import torch.nn as nn
import torch.optim as optim

# import torchviz

# Simulated data (A_i, ∇R(A_i))
A_data = torch.randn(50, 5)  # Random 2D tensor
true_theta = torch.tensor([2.0, 0.5])  # True parameters for R(A, theta)
true_gradients = true_theta[0] * true_theta[1] * torch.exp(true_theta[1] * A_data)  # Simulated ∇R(A_i)

# Define a learnable model for R(A, θ)
class RFunction(nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.tensor([1.0, 1.0]))  # Initialize parameters

    def forward(self, A):
        return self.theta[0] * torch.exp(self.theta[1] * A)  # Example function R(A, θ)

# Instantiate model
model = RFunction()
optimizer = optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()


# Training loop to fit ∇R(A, θ) to observed gradients
for epoch in range(1000):

    optimizer.zero_grad()

    # RE-CREATE A_data with requires_grad=True at every step
    A_data = A_data.detach().requires_grad_()

    # Compute predicted gradient ∇R(A, θ) via autograd
    R_values = model(A_data)
    gradients = torch.autograd.grad(R_values.sum(), A_data, create_graph=True)[0]

    # Compute loss
    loss = loss_fn(gradients, true_gradients)

    # print computational graph
    # torchviz.make_dot(loss, params=dict(model.named_parameters())).render(f"graph-{epoch}", format="png")
    
    # Backpropagate and update θ
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item()}")

# Learned parameters
print("Learned theta:", model.theta.detach().numpy())

Epoch 0: Loss = 10.189298629760742
Epoch 10: Loss = 0.9581233263015747
Epoch 20: Loss = 1.128289818763733
Epoch 30: Loss = 0.9216678142547607
Epoch 40: Loss = 0.4604668617248535
Epoch 50: Loss = 0.08537350594997406
Epoch 60: Loss = 0.10286110639572144
Epoch 70: Loss = 0.06661003828048706
Epoch 80: Loss = 0.04724550247192383
Epoch 90: Loss = 0.04019279405474663
Epoch 100: Loss = 0.03149869292974472
Epoch 110: Loss = 0.026242369785904884
Epoch 120: Loss = 0.02182808332145214
Epoch 130: Loss = 0.018094973638653755
Epoch 140: Loss = 0.015073939226567745
Epoch 150: Loss = 0.012560511007905006
Epoch 160: Loss = 0.010463815182447433
Epoch 170: Loss = 0.008718715980648994
Epoch 180: Loss = 0.007264194078743458
Epoch 190: Loss = 0.006050646770745516
Epoch 200: Loss = 0.0050376164726912975
Epoch 210: Loss = 0.004191874992102385
Epoch 220: Loss = 0.0034857105929404497
Epoch 230: Loss = 0.0028960786294192076
Epoch 240: Loss = 0.00240386207588017
Epoch 250: Loss = 0.0019931227434426546
Epoch 260: L

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Define your RFunction model (unchanged)
class RFunction(nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.tensor([1.0, 1.0]))  # Initialize parameters

    def forward(self, A):
        return self.theta[0] * torch.exp(self.theta[1] * A)  # Example function R(A, θ)

# For this example, we'll use the same simulated data, but in batches
A_data = torch.randn(500, 5)  # More data points
true_theta = torch.tensor([2.0, 0.5]) 
true_gradients = true_theta[0] * true_theta[1] * torch.exp(true_theta[1] * A_data)

# Create dataset and dataloader
dataset = TensorDataset(A_data, true_gradients)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Instantiate model
model = RFunction()
optimizer = optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

# Training loop with batches
num_epochs = 20

for epoch in range(num_epochs):
    epoch_loss = 0.0
    batch_count = 0
    
    for A_batch, true_gradients_batch in dataloader:
        optimizer.zero_grad()
        
        # Make inputs require gradients
        A_batch = A_batch.detach().requires_grad_()
        
        # Compute predicted gradient ∇R(A, θ) via autograd
        R_values = model(A_batch)
        gradients = torch.autograd.grad(R_values.sum(), A_batch, create_graph=True)[0]
        
        # Compute loss for this batch
        loss = loss_fn(gradients, true_gradients_batch)
        
        # Backpropagate and update θ
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        batch_count += 1
    
    avg_epoch_loss = epoch_loss / batch_count
    print(f"Epoch {epoch}: Average Loss = {avg_epoch_loss:.8f}")

# Learned parameters
print("Learned theta:", model.theta.detach().numpy())

Epoch 0: Average Loss = 0.34390883
Epoch 1: Average Loss = 0.08219035
Epoch 2: Average Loss = 0.03113064
Epoch 3: Average Loss = 0.01436420
Epoch 4: Average Loss = 0.00687402
Epoch 5: Average Loss = 0.00367019
Epoch 6: Average Loss = 0.00206267
Epoch 7: Average Loss = 0.00107634
Epoch 8: Average Loss = 0.00048235
Epoch 9: Average Loss = 0.00024246
Epoch 10: Average Loss = 0.00011804
Epoch 11: Average Loss = 0.00005754
Epoch 12: Average Loss = 0.00002767
Epoch 13: Average Loss = 0.00001268
Epoch 14: Average Loss = 0.00000546
Epoch 15: Average Loss = 0.00000260
Epoch 16: Average Loss = 0.00000125
Epoch 17: Average Loss = 0.00000036
Epoch 18: Average Loss = 0.00000014
Epoch 19: Average Loss = 0.00000005
Learned theta: [1.9992448 0.5001065]


In [27]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
dy_dx = torch.autograd.grad(y, x)  # Computes dy/dx
print(dy_dx)  # (tensor(4.0),)

(tensor(4.),)
